# SVM 5-fold cross-validation

Replaces the thesis §4.7 comparison table, which currently reads `1,0000` against
`1,0000` for both models and concludes they cannot be told apart.

Reuses `cnn-latest/results/splits/{variant}-fold{k}.csv` (70/10/20, fold seed `42+fold`),
so the SVM and CNN k-fold numbers are computed on identical rows. In-domain only —
matching `cnn-latest/results/kfold.csv` — and no model is saved.

The expectation is that this table stays saturated for both models. That is the point:
in-domain accuracy is a memorization check on this corpus (200 takes per class sharing
one keyboard, one preset, one voicing), so a random split puts near-copies of every test
clip in the training partition. The comparison that discriminates is out-of-domain, in
`train-variants.ipynb` and `compare.ipynb`.

In [ ]:
import subprocess, sys, time
from pathlib import Path
import pandas as pd

HERE = Path("/home/seya/code/chord-detection/training/notebooks/svm-latest")
PY = "/home/seya/code/chord-detection/.venv/bin/python"
sys.path.insert(0, str(HERE))
import svm_common as C

def fit_fold(variant, fold):
    t = time.time()
    p = subprocess.Popen([PY, str(HERE / "run_kfold.py"), "--variant", variant,
                          "--fold", str(fold)], cwd=HERE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line.rstrip(), flush=True)
    p.wait()
    print(f"[{variant}/fold{fold} exit {p.returncode} in {time.time() - t:.0f}s]")
    return p.returncode

for variant in ["orig", "clean"]:
    for fold in C.FOLDS:
        fit_fold(variant, fold)

In [ ]:
svm = pd.read_csv(C.OUT_DIR / "kfold.csv").assign(model="SVM")
cnn = pd.read_csv(C.CNN_DIR / "results" / "kfold.csv").assign(model="CNN")
both = pd.concat([cnn, svm], ignore_index=True)

tbl = both.pivot_table(index=["variant", "fold"], columns="model",
                       values=["test_accuracy", "test_macro_f1"])
summary = both.groupby(["model", "variant"])["test_accuracy"].agg(["mean", "std", "min"])
display(tbl.round(4))
display(summary.round(6))